# 02 – Beta, Cost of Equity and WACC

**Goal:** calculate the discount rate (WACC) used in the DCF model.

| Step | Formula |
|---|---|
| Beta | Covariance(AAPL return, S&P 500 return) / Variance(S&P 500 return) |
| Cost of Equity (CAPM) | Risk-free rate + Beta × Equity Risk Premium |
| Cost of Debt | Interest Expense / Total Debt |
| WACC | E/V × Cost of Equity + D/V × Cost of Debt × (1 − Tax Rate) |

In [1]:
import pandas as pd

price_long = pd.read_csv("../data/raw/price_data_long.csv", parse_dates=["date"])
prices = price_long.pivot(index="date", columns="ticker", values="close")[["AAPL", "^GSPC"]].sort_index()
assert prices.index.max() == pd.Timestamp("2026-11-13")
prices.tail()


ticker,AAPL,^GSPC
date,,
2026-11-09,350.423737,7549.939941
2026-11-10,348.245392,7520.930176
2026-11-11,355.359985,7619.379883
2026-11-12,354.880005,7592.399902
2026-11-13,356.269989,7561.669922


## 1. Beta

In [2]:
# daily return = (price today - price yesterday) / price yesterday
returns = prices.pct_change().dropna()

aapl_covariance = returns["AAPL"].cov(returns["^GSPC"])
variance = returns["^GSPC"].var()
beta_daily = aapl_covariance / variance
print("Beta (daily, 2015-Nov 13 2026):", round(beta_daily, 3))

Beta (daily, 2015-Nov 13 2026): 1.177


In [3]:
# Cross-check: Yahoo Finance uses 5 years of MONTHLY returns
monthly = prices.resample("ME").last().pct_change().dropna().tail(60)
beta_monthly = monthly["AAPL"].cov(monthly["^GSPC"]) / monthly["^GSPC"].var()
print("Beta (monthly, last 5 years):", round(beta_monthly, 3))

# Beta changes with the time window and frequency. The model uses the daily 2015-2026 beta (1.177).

Beta (monthly, last 5 years): 1.088


## 2. Cost of Equity (CAPM)

In [4]:
beta = round(beta_daily, 3)                 # daily beta calculated above (Jan 2015 - Nov 13 2026)
risk_free_rate = 0.0494      # 10-year US Treasury yield, Nov 1 2026 (Damodaran)
equity_risk_premium = 0.0441 # implied ERP for the S&P 500, Nov 1 2026 (Damodaran)

cost_of_equity = risk_free_rate + beta * equity_risk_premium
print(f"Cost of Equity: {cost_of_equity:.2%}")

Cost of Equity: 10.13%


## 3. Cost of Debt and Tax Rate (from the 10-K data)

In [5]:
income = pd.read_csv("../data/income_statement.csv").set_index("fiscal_year")
balance = pd.read_csv("../data/balance_sheet.csv").set_index("fiscal_year")

# Apple stopped reporting Interest Expense after FY2023 -> use FY2023 (latest available)
cost_of_debt = income.loc[2023, "interest_expense"] / balance.loc[2023, "total_debt"]

# Effective tax rate, latest year
tax_rate = income.loc[2026, "tax_provision"] / income.loc[2026, "pretax_income"]
after_tax_cost_of_debt = cost_of_debt * (1 - tax_rate)

print(f"Cost of Debt (pre-tax, FY2023): {cost_of_debt:.2%}")
print(f"Tax Rate (FY2026): {tax_rate:.2%}")
print(f"After-tax Cost of Debt: {after_tax_cost_of_debt:.2%}")

Cost of Debt (pre-tax, FY2023): 3.54%
Tax Rate (FY2026): 16.99%
After-tax Cost of Debt: 2.94%


## 4. WACC

In [6]:
share_price = 356.27          # AAPL close, Nov 13 2026
shares_outstanding = 14532.182 # million shares, Oct 23 2026 (10-K cover page)
total_debt = balance.loc[2026, "total_debt"]   # $ millions, FY2026

market_cap = share_price * shares_outstanding   # E ($ millions)
total_value = market_cap + total_debt           # V = E + D
weight_equity = market_cap / total_value
weight_debt = total_debt / total_value

WACC = weight_equity * cost_of_equity + weight_debt * after_tax_cost_of_debt
print(f"E/V: {weight_equity:.2%} | D/V: {weight_debt:.2%}")
print(f"WACC: {WACC:.2%}")

E/V: 98.48% | D/V: 1.52%
WACC: 10.02%


**Reading the result:** debt is only ~1.5% of Apple's capital, so WACC (~10.1%) is almost the same as the Cost of Equity. Any project Apple invests in must earn at least ~10% a year to create value for shareholders.

### Save the results
Notebook 03 reads this file, so the DCF uses exactly the same WACC.

In [7]:
wacc_table = pd.DataFrame({
    "metric": ["beta", "cost_of_equity", "after_tax_cost_of_debt", "weight_equity", "wacc"],
    "value": [beta, cost_of_equity, after_tax_cost_of_debt, weight_equity, WACC],
})
wacc_table.to_csv("../data/wacc.csv", index=False)
wacc_table

,metric,value
0,beta,1.177000
1,cost_of_equity,0.101306
2,after_tax_cost_of_debt,0.029387
3,weight_equity,0.984763
4,wacc,0.100210
